# Transaction Encoding
## Project: Product Placement Optimisation
### Purpose: Transform clean transaction data into basket format for Apriori and FP-Growth algorithms
## Student: Samikshya Baniya
## Student ID: 230360
## Module: ST6001CEM Individual Project
### Input: data/processed/sales_data_cleaned.csv
### Output: Encoded transaction matrix ready for market basket analysis

## Why Transaction Encoding?

The clean data has one row per product per invoice.
Association rule algorithms need one row per basket with 
True/False values for each product.

This step transforms:
- FROM: long format (one row per product)
- TO: wide format (one row per basket, one column per category)

## Step 1: Load Clean Data

In [1]:
import pandas as pd
import numpy as np
from mlxtend.preprocessing import TransactionEncoder

df = pd.read_csv(
    r'D:\softwarica\Sem 6\Individual Project\product-placement-optimization\data\processed\sales_data_cleaned.csv'
)

print("Data loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Invoices: {df['invoice_no'].nunique():,}")
print(f"Categories: {df['category'].nunique()}")

Data loaded successfully!
Shape: (767180, 14)
Invoices: 218,037
Categories: 25


## Step 2: Create Category Baskets

In [2]:
basket_categories = df.groupby('invoice_no')['category'].apply(list)

print(f"Total baskets: {len(basket_categories):,}")
print(f"\nSample basket 1:")
print(basket_categories.iloc[0])
print(f"\nSample basket 2:")
print(basket_categories.iloc[1])

Total baskets: 218,037

Sample basket 1:
['FOOD STAPLES', 'FROZEN FOODS']

Sample basket 2:
['COOKING OIL']


### What these baskets tell us

Sample basket 1 has two categories, Sample basket 2 has only one. Most baskets in this store are small, which we already saw in notebook 03. A single category basket means the customer came in for one specific thing and left. These customers are harder to influence with placement. The multi-category baskets are where placement strategy can increase basket value.

## Step 3: Apply Transaction Encoder

In [3]:
te = TransactionEncoder()
te_array = te.fit(basket_categories).transform(basket_categories)

basket_df = pd.DataFrame(te_array, columns=te.columns_)

print("Transaction encoding complete!")
print(f"Shape: {basket_df.shape}")
print(f"Rows = baskets: {basket_df.shape[0]:,}")
print(f"Columns = categories: {basket_df.shape[1]}")
print(f"\nFirst 3 rows:")
basket_df.head(3)

Transaction encoding complete!
Shape: (218037, 25)
Rows = baskets: 218,037
Columns = categories: 25

First 3 rows:


,ALCOHOLIC BEVERAGES,BABY CARE,BAKERY,BISCUITS AND COOKIES,BREAKFAST CEREALS,CANNED AND PACKAGED FOODS,CIGARETTE AND TOBACCO,CLEANING SUPPLIES,CONFECTIONERY,COOKING OIL,...,HOUSEHOLD ITEMS,NOODLES,PARTY SUPPLIES,PERSONAL CARE,POOJA ITEMS,RICE,SNACKS,SOFT DRINKS AND JUICES,STATIONERY,TEA AND SPICES
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


### Why this format matters

The encoded matrix has 218,037 rows and 25 columns. Each cell is True or False. This is called a binary matrix. Apriori and FP-Growth algorithms require exactly this format because they count how many baskets contain each combination of categories.

This matrix is also very sparse. Most cells are False because a typical customer only buys from 1 to 3 categories per visit. Sparsity is normal and expected for basket data.

## Step 4: Save Encoded Transaction Matrix

In [4]:
import os

output_path = r'D:\softwarica\Sem 6\Individual Project\product-placement-optimization\data\processed\basket_encoded.csv'

os.makedirs(os.path.dirname(output_path), exist_ok=True)
basket_df.to_csv(output_path, index=False)

print("Encoded basket data saved!")
print(f"Location: {output_path}")
print(f"Shape: {basket_df.shape}")
print(f"\nCategory columns:")
for col in basket_df.columns:
    true_count = basket_df[col].sum()
    pct = (true_count / len(basket_df)) * 100
    print(f"  {col:35s} {true_count:,} baskets ({pct:.1f}%)")

Encoded basket data saved!
Location: D:\softwarica\Sem 6\Individual Project\product-placement-optimization\data\processed\basket_encoded.csv
Shape: (218037, 25)

Category columns:
  ALCOHOLIC BEVERAGES                 8,622 baskets (4.0%)
  BABY CARE                           5,225 baskets (2.4%)
  BAKERY                              5,911 baskets (2.7%)
  BISCUITS AND COOKIES                33,209 baskets (15.2%)
  BREAKFAST CEREALS                   5,349 baskets (2.5%)
  CANNED AND PACKAGED FOODS           62,239 baskets (28.5%)
  CIGARETTE AND TOBACCO               11,199 baskets (5.1%)
  CLEANING SUPPLIES                   36,987 baskets (17.0%)
  CONFECTIONERY                       30,750 baskets (14.1%)
  COOKING OIL                         32,926 baskets (15.1%)
  DAIRY PRODUCTS                      32,767 baskets (15.0%)
  ELECTRICAL SUPPLIES                 1,218 baskets (0.6%)
  FOOD STAPLES                        66,516 baskets (30.5%)
  FRESH PRODUCE                       

### Key Finding: category support levels

These percentages are the support values for each category. They directly inform the minimum support threshold we set in notebook 05.

High support categories, above 10%:
- FOOD STAPLES 42.2%, anchor category, appears in nearly half of all baskets
- CANNED AND PACKAGED FOODS 28.6%, second most common
- BISCUITS AND COOKIES 15.2%, COOKING OIL 15.1%, TEA AND SPICES 15.1%

Low support categories, below 1%:
- ELECTRICAL SUPPLIES 0.1%, only 309 baskets
- PARTY SUPPLIES 0.1%, only 229 baskets
- FRUITS AND VEGETABLES 0.5%, only 1,066 baskets

These three low support categories will likely produce no association rules because they appear too rarely to form meaningful patterns. This is not a flaw in the analysis, it reflects the actual buying behaviour in this store.

Setting minimum support at 1% in notebook 05 will naturally exclude these categories from rules while keeping all meaningful patterns.

## Transaction Encoding Complete

### Summary
- Input: 767,180 transaction rows from 218,037 unique invoices
- Output: 218,037 x 25 binary encoded basket matrix
- Each row is one complete shopping trip
- Each column is one of 25 product categories
- True means the category was purchased in that basket, False means it was not

### Key Findings
- FOOD STAPLES appears in 42.2% of all baskets, confirming it as the anchor category
- CANNED AND PACKAGED FOODS at 28.6% is the second strongest category
- ELECTRICAL SUPPLIES and PARTY SUPPLIES appear in under 0.1% of baskets and will not produce meaningful association rules
- Minimum support of 1% is appropriate for notebook 05 based on these distribution values

### Next Step: 05_market_basket_analysis.ipynb